In [1]:

# ============================================================
# CAMPAIGN ORDERS RECOMMENDATION - TRAINING PIPELINE
# Target: orders_attributed
# ============================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. PATHS
# ============================================================

DATA_PATH = Path("data/marketing_campaigns.csv")
ARTIFACT_DIR = Path("artifacts")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "campaign_orders_model.joblib"
HISTORY_PATH = ARTIFACT_DIR / "campaign_history.csv"

In [2]:



# ============================================================
# 2. GOVERNORATE MAPPING
# ============================================================

GOVERNORATE_MAP = {
    "القاهرة": "Cairo",
    "الجيزة": "Giza",
    "الإسكندرية": "Alexandria",
    "الاسكندرية": "Alexandria",
    "القليوبية": "Qalyubia",
    "البحيرة": "Beheira",
    "الغربية": "Gharbia",
    "المنوفية": "Monufia",
    "كفر الشيخ": "Kafr El Sheikh",
    "الدقهلية": "Dakahlia",
    "دمياط": "Damietta",
    "الشرقية": "Sharqia",
    "بورسعيد": "Port Said",
    "الإسماعيلية": "Ismailia",
    "السويس": "Suez",
    "شمال سيناء": "North Sinai",
    "جنوب سيناء": "South Sinai",
    "بني سويف": "Beni Suef",
    "الفيوم": "Faiyum",
    "المنيا": "Minya",
    "أسيوط": "Assiut",
    "سوهاج": "Sohag",
    "قنا": "Qena",
    "الأقصر": "Luxor",
    "أسوان": "Aswan",
    "البحر الأحمر": "Red Sea",
    "الوادي الجديد": "New Valley",
    "مطروح": "Matrouh",
}


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

required_columns = [
    "campaign_id",
    "start_date",
    "end_date",
    "channel",
    "target_audience",
    "governorate_ar",
    "budget_EGP",
    "spend_EGP",
    "impressions",
    "clicks",
    "orders_attributed",
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df = df[required_columns].copy()


# ============================================================
# 4. DATA TYPES
# ============================================================

df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")

numeric_columns = [
    "budget_EGP",
    "spend_EGP",
    "impressions",
    "clicks",
    "orders_attributed",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_columns = [
    "channel",
    "target_audience",
    "governorate_ar",
]

for col in categorical_columns:
    df[col] = df[col].astype("string").str.strip()


# ============================================================
# 5. GOVERNORATE NORMALIZATION
# ============================================================

df["governorate_ar"] = df["governorate_ar"].str.strip()

df["governorate"] = (
    df["governorate_ar"]
    .map(GOVERNORATE_MAP)
    .fillna(df["governorate_ar"])
)


# ============================================================
# 6. FEATURE ENGINEERING
# ============================================================

df["duration_days"] = (
    df["end_date"] - df["start_date"]
).dt.days + 1

df["start_month"] = df["start_date"].dt.month
df["start_dayofweek"] = df["start_date"].dt.dayofweek

df["end_month"] = df["end_date"].dt.month


# ============================================================
# 7. HISTORICAL CAMPAIGN METRICS
# ============================================================

df["cost_per_attributed_order"] = np.where(
    df["orders_attributed"] > 0,
    df["spend_EGP"] / df["orders_attributed"],
    np.nan
)

df["CTR"] = np.where(
    df["impressions"] > 0,
    df["clicks"] / df["impressions"],
    np.nan
)

df["conversion_rate"] = np.where(
    df["clicks"] > 0,
    df["orders_attributed"] / df["clicks"],
    np.nan
)


# ============================================================
# 8. SAVE HISTORICAL DATA
# ============================================================

df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")


# ============================================================
# 9. MODELING DATA
# ============================================================

FEATURES = [
    "channel",
    "target_audience",
    "governorate",
    "budget_EGP",
    "duration_days",
    "start_month",
    "start_dayofweek",
    "end_month",
]

TARGET = "orders_attributed"

model_df = df.dropna(
    subset=FEATURES + [TARGET]
).copy()

model_df = model_df[
    (model_df["budget_EGP"] >= 0) &
    (model_df["duration_days"] > 0) &
    (model_df[TARGET] >= 0)
].copy()

if len(model_df) < 10:
    raise ValueError(
        "Not enough valid campaign records for a reliable train/test split."
    )

X = model_df[FEATURES]
y = model_df[TARGET]


# ============================================================
# 10. PREPROCESSING
# ============================================================

CAT_FEATURES = [
    "channel",
    "target_audience",
    "governorate",
]

NUM_FEATURES = [
    "budget_EGP",
    "duration_days",
    "start_month",
    "start_dayofweek",
    "end_month",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CAT_FEATURES,
        ),
        (
            "numerical",
            "passthrough",
            NUM_FEATURES,
        ),
    ]
)


# ============================================================
# 11. MODEL
# ============================================================

regressor = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.9,
    random_state=42,
    n_jobs=-1,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ]
)


# ============================================================
# 12. TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("=" * 55)
print("CAMPAIGN ORDERS MODEL EVALUATION")
print("=" * 55)
print(f"Training records : {len(X_train)}")
print(f"Testing records  : {len(X_test)}")
print(f"MAE              : {mae:.3f}")
print(f"RMSE             : {rmse:.3f}")
print(f"R2               : {r2:.3f}")


# ============================================================
# 13. SAVE EVERYTHING REQUIRED FOR INFERENCE
# ============================================================

artifact = {
    "model": model,
    "features": FEATURES,
    "categorical_features": CAT_FEATURES,
    "numerical_features": NUM_FEATURES,
    "channels": sorted(
        df["channel"].dropna().astype(str).unique().tolist()
    ),
    "target_audiences": sorted(
        df["target_audience"].dropna().astype(str).unique().tolist()
    ),
    "governorate_map": GOVERNORATE_MAP,
    "global_median_budget": float(df["budget_EGP"].median()),
    "evaluation": {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    },
}

joblib.dump(artifact, MODEL_PATH)

print("\nSaved model:", MODEL_PATH)
print("Saved historical data:", HISTORY_PATH)


CAMPAIGN ORDERS MODEL EVALUATION
Training records : 1760
Testing records  : 440
MAE              : 8585.983
RMSE             : 12969.020
R2               : 0.191

Saved model: artifacts\campaign_orders_model.joblib
Saved historical data: artifacts\campaign_history.csv
